# 感情AI: 第一段階(内部状態の浸透と滲み)— Kaggle 版(Save & Run All 専用)

Colab の GPU 上限待ちの代替として Kaggle(週 30 時間、1 セッション 12 時間)を使う。Colab 版と**同じスクリプト**
(`run_c0.py`, `run_g1.py`, `analyze_granularity.py`, `leak_probe.py`)を呼ぶだけで、コードは変えない。

**このノートブックは固定**。何を実行するか(段階・seed 数・更新回数など)は、リポジトリの
`llm_grounding/run_config.json` から読む。切り替えは「`run_config.json` を直して push → 同じノートブックを
**Save Version → Save & Run All (Commit)**」だけ。対話セッションでのセル実行は使わない。

- 段階(`stage`): `g1_smoke`(G1 の試走)/ `g1`(関門 G1、3 seed)/ `c0`(C0 本番)/ `c0prime` / `t1`(未実装。止まる)
- 結果は `/kaggle/working/results/<stage>/` に保存し、最後のセルで要約(判定・主要数値)をログに print する。
  ファイルの取り出しは節目だけ(Output タブから `results.zip`。SHA-256 一覧もログに出る)。

## 初回の設定(1 回だけ)

1. 電話番号の確認(Settings → Phone verification)。GPU と Internet に必要。
2. **Create → New Notebook → File → Import Notebook** で GitHub の URL
   `https://github.com/seina369/homeostatic-agent-experiments/blob/main/llm_grounding/Kaggle_run_stage1.ipynb` を読み込む。
3. 右側の **Session options**: **Accelerator = GPU T4 x2**(なければ P100)、**Internet = On**、Persistence は任意(Files only)。
4. 右上 **Save Version → Save & Run All (Commit)**。終わったら版のログ(Logs)を読む。


## 1. 準備: GPU 確認、GitHub から clone、依存、環境の検算

GPU がなければここで止める。Kaggle 同梱の `torchao` が peft の要求より古いと LoRA の付与で失敗するので外す。
`run_config.json` の `run_tests` が true なら GPU 不要のテスト(GRPO・注入器・読み取り器・G1 ランナー)を回す。

In [ ]:
import os, sys, json, subprocess, time
import torch

T_START = time.time()
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPU数:", torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError("GPU が見つからない。Session options → Accelerator を「GPU T4 x2」か「GPU P100」にしてから Save & Run All をやり直す。")
for i in range(torch.cuda.device_count()):
    print(f"GPU{i}:", torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

REPO_URL = "https://github.com/seina369/homeostatic-agent-experiments.git"
REPO_DIR = "/kaggle/working/repo"
RESULTS_ROOT = "/kaggle/working/results"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
LLM_DIR = os.path.join(REPO_DIR, "llm_grounding")
os.makedirs(RESULTS_ROOT, exist_ok=True)
print("commit:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

with open(os.path.join(LLM_DIR, "run_config.json"), encoding="utf-8") as f:
    CFG = json.load(f)
print("run_config.json:", json.dumps({k: v for k, v in CFG.items() if not k.startswith("_")}, ensure_ascii=False))

r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(LLM_DIR, "requirements.txt")], capture_output=True, text=True)
print("pip -r requirements.txt rc", r.returncode, r.stderr[-400:] if r.returncode else "")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True, text=True)
import transformers, peft, sklearn
print("transformers", transformers.__version__, "| peft", peft.__version__, "| scikit-learn", sklearn.__version__)

if CFG.get("run_tests", True):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", "-p", "no:cacheprovider",
                        "test_grpo.py", "test_state_injector.py", "test_leak_probe.py", "test_run_g1.py"],
                       cwd=LLM_DIR, capture_output=True, text=True)
    print(r.stdout[-3000:]); print(r.stderr[-1000:])
    if r.returncode != 0:
        raise RuntimeError("GPU 不要のテストが Kaggle 同梱のバージョンで落ちた。上のログを見て止まる。")
print(f"準備 完了 {(time.time() - T_START) / 60:.1f} 分")


## 2. 動作確認

環境定数が事前登録の確定値であること、モデルが読み込めて `respond()` が 1 回動くこと(Hugging Face から取得)。

In [ ]:
sys.path.insert(0, LLM_DIR)
import emotion_grounding_env as E
from seed_records import env_constants
from torch_qwen_policy import TorchPolicy

print("環境定数:", env_constants())
assert (E.B0, E.BUDGET_LOW_THRESHOLD, E.U_OPT, E.U_MIN, E.U_MAX) == (340, 85.0, 0.7, 0.0, 2.5), "事前登録の確定値と違う"
assert E.PROMPT_VERSION == 2, "v2 のプロンプトになっていない"
t0 = time.time()
_p = TorchPolicy(verbose=False)
print(f"モデル読み込み {time.time() - t0:.0f} 秒")
_r = _p.respond(E.PROMPT_TEMPLATE.format(task="What is 23 + 19?", budget=340, error=0, uncertainty=0.70))
print(repr(_r.text[:160]), f"n_tokens={_r.n_tokens} mean_entropy={_r.mean_entropy:.3f}")
print(f"GPU0 使用中(解放前) {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
# 動作確認で読んだモデルを解放してから本番のランナーに渡す(同じプロセスで動かすため)
import gc
del _p, _r
gc.collect()
torch.cuda.empty_cache()
print(f"GPU0 使用中(解放後) {torch.cuda.memory_allocated(0) / 1e9:.2f} GB、確保済み {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")


## 3. 実行(`run_config.json` の stage に従う)

- `g1_smoke` / `g1`: `run_g1.py`(G1 のランナー)。`seeds`, `seed_start`, `updates`, `eval_episodes`, `groups_per_update`, `G`, `lr`, `beta_kl`, `micro_batch` を渡す。
- `c0`: `run_c0.py --seeds --episodes` → `analyze_granularity.py` と `leak_probe.py`。
- `c0prime` / `t1`: 未実装。メッセージを出して止まる。
`extra_args` はそのままランナーに追加で渡す。

In [ ]:
STAGE = CFG["stage"]
OUT_DIR = os.path.join(RESULTS_ROOT, STAGE)
os.makedirs(OUT_DIR, exist_ok=True)
extra = list(CFG.get("extra_args", []))
t0 = time.time()

import runpy

def run(script, argv):
    # スクリプトをこのプロセス(カーネル)で実行する。subprocess で起動すると Kaggle ではモデル読み込みの
    # 途中で止まったため(2026-09-13、版 #1・#2)。print はそのままログに流れる。戻り値は終了コード。
    print("$", script, " ".join(argv), flush=True)
    old_argv = sys.argv
    sys.argv = [script] + list(argv)
    try:
        runpy.run_path(os.path.join(LLM_DIR, script), run_name="__main__")
        return 0
    except SystemExit as e:
        return 0 if e.code in (None, 0) else (e.code if isinstance(e.code, int) else 1)
    finally:
        sys.argv = old_argv
        sys.stdout.flush()

import gc
gc.collect(); torch.cuda.empty_cache()
_alloc = torch.cuda.memory_allocated(0) / 1e9
print(f"GPU0 使用中 {_alloc:.2f} GB、確保済み {torch.cuda.memory_reserved(0) / 1e9:.2f} GB(ランナー呼び出し直前)", flush=True)
if _alloc > 0.5:
    raise RuntimeError(f"動作確認のモデルが解放されていない(GPU0 使用中 {_alloc:.2f} GB)。")
if STAGE in ("g1_smoke", "g1"):
    rc = run("run_g1.py", ["--out-dir", OUT_DIR,
           "--seeds", str(CFG["seeds"]), "--seed-start", str(CFG.get("seed_start", 0)),
           "--updates", str(CFG["updates"]), "--eval-episodes", str(CFG["eval_episodes"]),
           "--groups-per-update", str(CFG.get("groups_per_update", 4)), "--G", str(CFG.get("G", 8)),
           "--lr", str(CFG.get("lr", 1e-5)), "--beta-kl", str(CFG.get("beta_kl", 0.04)),
           "--micro-batch", str(CFG.get("micro_batch", 0))] + extra)
elif STAGE == "c0":
    rc = run("run_c0.py", ["--policy", "torch", "--seeds", str(CFG["seeds"]),
              "--seed-start", str(CFG.get("seed_start", 0)), "--episodes", str(CFG.get("episodes", 30)),
              "--temperature", "1.0", "--out-dir", OUT_DIR] + extra)
    if rc == 0:
        run("analyze_granularity.py", ["--dir", OUT_DIR, "--out", os.path.join(OUT_DIR, "c0_analysis.json")])
        run("leak_probe.py", ["--dir", OUT_DIR, "--out", os.path.join(OUT_DIR, "c0_leak_probe.json")])
elif STAGE in ("c0prime", "t1"):
    raise RuntimeError(f"stage={STAGE} はまだ実装されていない(事前登録 3 章・6 章 G4 の設計に従って実装してから)。")
else:
    raise RuntimeError(f"未知の stage: {STAGE}")
print(f"stage={STAGE} rc={rc} 所要 {(time.time() - t0) / 60:.1f} 分", flush=True)
if rc != 0:
    raise RuntimeError(f"stage={STAGE} のランナーが rc={rc} で終了した(上のログを見る)")


## 4. 要約(判定・主要数値)と保存状況

最後にログで読めるように、結果の JSON から要点だけを print する。`/kaggle/working/results` を zip にまとめ、SHA-256 の一覧を出す。

In [ ]:
import glob, hashlib, zipfile, math

def fmt(x, nd=3):
    return "nan" if x is None or (isinstance(x, float) and math.isnan(x)) else (f"{x:.{nd}f}" if isinstance(x, float) else str(x))

print("=" * 70); print(f"要約 stage={STAGE} out={OUT_DIR}")
if STAGE in ("g1_smoke", "g1"):
    for p in sorted(glob.glob(os.path.join(OUT_DIR, "g1_seed*.json"))):
        with open(p, encoding="utf-8") as f:
            r = json.load(f)
        eb, ea, j, el = r["eval_before"], r["eval_after"], r["judgement"], r["elapsed_seconds"]
        print(f"seed {r['seed']}: 平均逸脱 {fmt(eb['mean_deviation'])} → {fmt(ea['mean_deviation'])} "
              f"(相対低下 {fmt(j['rel_drop'])}) 正答率 {fmt(eb['correct_rate'], 2)} → {fmt(ea['correct_rate'], 2)} "
              f"書式不履行 {fmt(eb['format_fail_rate'], 2)} → {fmt(ea['format_fail_rate'], 2)} "
              f"トークン {fmt(eb['mean_tokens'], 1)} → {fmt(ea['mean_tokens'], 1)}")
        print(f"        所要 評価前 {el['eval_before'] / 60:.1f} 分 / 学習 {el['train'] / 60:.1f} 分 / 評価後 {el['eval_after'] / 60:.1f} 分 / 合計 {el['total'] / 60:.1f} 分 | "
              f"1 グループ {fmt(r.get('seconds_per_group'), 1)} 秒 | GPU メモリ最大 {fmt(r.get('peak_gpu_memory_gb'), 2)} GB | NaN={r['nan_found']} | 学習対象 {r['n_trainable_params']:,}")
        tr = r["training"]
        if tr:
            step = max(1, len(tr) // 8)
            print("        更新ごと(update: 平均報酬 / KL / 損失):", "; ".join(
                f"{e['update']}: {e['mean_reward']:.3f} / {e['kl']:.4f} / {e['loss']:.4f}" for e in tr[::step]))
        print(f"        判定: {'合格' if j['pass'] else '不合格'} {j['reasons']}")
    sp = os.path.join(OUT_DIR, "g1_summary.json")
    if os.path.exists(sp):
        with open(sp, encoding="utf-8") as f:
            s = json.load(f)
        print(f"G1 全体: {'合格' if s['pass'] else '不合格'}({s['n_pass']}/{s['n_seeds']} seed 合格、必要 {s['required_seeds']})")
elif STAGE == "c0":
    files = sorted(glob.glob(os.path.join(OUT_DIR, "c0_seed*.json")))
    print(f"{len(files)} seed 保存済み")
    for f in files:
        with open(f, encoding="utf-8") as fh:
            p = json.load(fh)
        print(f"  {os.path.basename(f)}: complete={p.get('complete')} episodes={p.get('n_episodes')} steps={p.get('n_steps')} elapsed={p.get('elapsed_seconds', 0) / 60:.1f}min")
    for name in ("c0_analysis.json", "c0_leak_probe.json"):
        pth = os.path.join(OUT_DIR, name)
        if os.path.exists(pth):
            with open(pth, encoding="utf-8") as fh:
                d = json.load(fh)
            print(name, "summary:", json.dumps(d.get("summary", d.get("overall", {})), ensure_ascii=False)[:800])

zip_path = "/kaggle/working/results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, names in os.walk(RESULTS_ROOT):
        for n in names:
            full = os.path.join(root, n)
            z.write(full, arcname=os.path.relpath(full, "/kaggle/working"))
            print(os.path.relpath(full, "/kaggle/working"), os.path.getsize(full), hashlib.sha256(open(full, "rb").read()).hexdigest())
print("ZIP", os.path.getsize(zip_path), hashlib.sha256(open(zip_path, "rb").read()).hexdigest())
print(f"全体 所要 {(time.time() - T_START) / 60:.1f} 分")
